# 📊 Soal 2: Exploratory Data Analysis & Preprocessing
## Dataset: Obesity Levels (ObesityDataSet.csv)
**Mata Kuliah:** Pembelajaran Mesin — UAS Genap 2025/2026

---
Dataset ini berisi data kebiasaan makan, aktivitas fisik, dan kondisi fisik individu untuk mengklasifikasikan **tingkat obesitas** ke dalam 7 kategori:
- Insufficient_Weight
- Normal_Weight
- Overweight_Level_I & II
- Obesity_Type_I, II & III

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('Set2')

# Load dataset
DATA_PATH = '../data/ObesityDataSet.csv'
OUTPUT_PATH = '../outputs/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Analisis Kualitas Data

In [ ]:
print('=== INFO DATASET ===')
df.info()
print()
print('=== STATISTIK DESKRIPTIF ===')
df.describe()

In [ ]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Tidak ada missing values!')

print('\n=== DUPLIKAT ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

print('\n=== TIPE DATA ===')
print(df.dtypes)

In [ ]:
# Cek inkonsistensi nilai kategorikal
cat_cols = df.select_dtypes(include='object').columns
print('=== NILAI UNIK VARIABEL KATEGORIKAL ===')
for col in cat_cols:
    print(f'{col}: {df[col].unique()}')

## 3. Analisis Univariat

In [ ]:
# Distribusi target
fig, ax = plt.subplots(figsize=(12, 5))
order = df['NObeyesdad'].value_counts().index
sns.countplot(data=df, x='NObeyesdad', order=order, ax=ax, palette='Set2')
ax.set_title('Distribusi Kelas Target (NObeyesdad)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tingkat Obesitas')
ax.set_ylabel('Jumlah')
plt.xticks(rotation=30, ha='right')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}01_distribusi_target.png', dpi=150)
plt.show()
print('Dataset ini seimbang — setiap kelas memiliki 72 sampel.')

In [ ]:
# Distribusi fitur numerik
num_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('Nilai')
    axes[i].set_ylabel('Frekuensi')
plt.suptitle('Distribusi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}02_distribusi_numerik.png', dpi=150)
plt.show()

In [ ]:
# Distribusi fitur kategorikal
cat_feat = ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(cat_feat):
    vc = df[col].value_counts()
    axes[i].bar(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)
plt.suptitle('Distribusi Fitur Kategorikal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}03_distribusi_kategorikal.png', dpi=150)
plt.show()

## 4. Analisis Multivariat & Insight

In [ ]:
# Insight 1: BMI per kategori obesitas
df['BMI'] = df['Weight'] / (df['Height'] ** 2)
order_cat = ['Insufficient_Weight','Normal_Weight','Overweight_Level_I',
             'Overweight_Level_II','Obesity_Type_I','Obesity_Type_II','Obesity_Type_III']

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='NObeyesdad', y='BMI', order=order_cat, ax=ax, palette='RdYlGn_r')
ax.set_title('Insight 1: Distribusi BMI per Kategori Obesitas', fontsize=14, fontweight='bold')
ax.set_xlabel('Kategori Obesitas')
ax.set_ylabel('BMI (kg/m²)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}04_insight1_bmi_per_kategori.png', dpi=150)
plt.show()
print('✅ Insight 1: BMI meningkat secara konsisten seiring tingkat obesitas. Ini mengonfirmasi validitas label.')

In [ ]:
# Insight 2: Korelasi antar fitur numerik
num_df = df[num_cols + ['BMI']]
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Insight 2: Heatmap Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}05_insight2_heatmap_korelasi.png', dpi=150)
plt.show()
print('✅ Insight 2: Weight dan BMI berkorelasi sangat tinggi. Age memiliki korelasi moderat dengan Weight.')

In [ ]:
# Insight 3: Riwayat keluarga vs kategori obesitas
ct = pd.crosstab(df['NObeyesdad'], df['family_history_with_overweight'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ct_pct.loc[order_cat].plot(kind='bar', ax=ax, color=['#FF6B6B','#4ECDC4'], width=0.7)
ax.set_title('Insight 3: Riwayat Keluarga Overweight per Kategori Obesitas (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Kategori Obesitas')
ax.set_ylabel('Persentase (%)')
ax.legend(['Tidak', 'Ya'], title='Riwayat Keluarga')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}06_insight3_riwayat_keluarga.png', dpi=150)
plt.show()
print('✅ Insight 3: Mayoritas penderita Obesity Type I-III memiliki riwayat keluarga overweight (>80%).')

In [ ]:
# Insight 4: Aktivitas fisik vs kategori obesitas
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df, x='NObeyesdad', y='FAF', order=order_cat, ax=ax,
               palette='muted', inner='box')
ax.set_title('Insight 4: Frekuensi Aktivitas Fisik per Kategori Obesitas', fontsize=13, fontweight='bold')
ax.set_xlabel('Kategori Obesitas')
ax.set_ylabel('FAF (Frekuensi Aktivitas Fisik, 0-3)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}07_insight4_aktivitas_fisik.png', dpi=150)
plt.show()
print('✅ Insight 4: Obesity Type III cenderung memiliki aktivitas fisik lebih rendah dibanding Normal/Insufficient Weight.')

In [ ]:
# Insight 5: Scatter plot Age vs BMI
palette_map = {
    'Insufficient_Weight': '#2196F3',
    'Normal_Weight': '#4CAF50',
    'Overweight_Level_I': '#FFC107',
    'Overweight_Level_II': '#FF9800',
    'Obesity_Type_I': '#FF5722',
    'Obesity_Type_II': '#E91E63',
    'Obesity_Type_III': '#9C27B0'
}
fig, ax = plt.subplots(figsize=(12, 7))
for cat, color in palette_map.items():
    subset = df[df['NObeyesdad'] == cat]
    ax.scatter(subset['Age'], subset['BMI'], c=color, label=cat, alpha=0.7, s=50)
ax.set_title('Insight 5: Hubungan Usia dan BMI berdasarkan Kategori Obesitas', fontsize=13, fontweight='bold')
ax.set_xlabel('Usia (tahun)')
ax.set_ylabel('BMI (kg/m²)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}08_insight5_age_vs_bmi.png', dpi=150)
plt.show()
print('✅ Insight 5: BMI tinggi tersebar di semua kelompok usia, namun Obesity Type III lebih banyak pada usia 20-40 tahun.')

## 5. Deteksi Outlier

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col)
plt.suptitle('Deteksi Outlier — Boxplot Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}09_outlier_boxplot.png', dpi=150)
plt.show()

# IQR outlier count
print('\n=== JUMLAH OUTLIER (IQR Method) ===')
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'  {col}: {len(outliers)} outlier ({len(outliers)/len(df)*100:.1f}%)')

## 6. Feature Engineering & Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib

df_clean = df.copy()

# Feature engineering: tambah BMI
df_clean['BMI'] = df_clean['Weight'] / (df_clean['Height'] ** 2)
print('✅ Fitur BMI ditambahkan.')

In [ ]:
# Encoding variabel kategorikal (Label Encoding)
# Justifikasi: target multi-kelas dan sebagian fitur biner → LabelEncoder efisien
le_dict = {}
cat_cols_to_encode = ['Gender', 'family_history_with_overweight', 'FAVC',
                       'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

for col in cat_cols_to_encode:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    le_dict[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Encode target
le_target = LabelEncoder()
df_clean['NObeyesdad_encoded'] = le_target.fit_transform(df_clean['NObeyesdad'])
print(f'\nTarget classes: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

In [ ]:
# Pisahkan fitur dan target
feature_cols = ['Gender','Age','Height','Weight','family_history_with_overweight',
                'FAVC','FCVC','NCP','CAEC','SMOKE','CH2O','SCC','FAF','TUE','CALC','MTRANS','BMI']

X = df_clean[feature_cols]
y = df_clean['NObeyesdad_encoded']

print(f'Fitur: {X.shape[1]} kolom')
print(f'Sampel: {X.shape[0]} baris')
print(f'Kelas target: {y.nunique()} kelas')

In [ ]:
# Train-Validation-Test Split (60-20-20)
# Justifikasi: dataset 504 sampel — 20% test (≈101 sampel) cukup untuk evaluasi yang representatif
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4,
                                                     random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5,
                                                  random_state=42, stratify=y_temp)

print(f'Train set  : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Val set    : {X_val.shape[0]} sampel ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test set   : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# StandardScaler untuk model yang sensitif terhadap skala (KNN)
# Justifikasi: fitur seperti Weight (kg) dan Height (m) memiliki skala berbeda
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print('✅ StandardScaler diterapkan pada train, val, test set.')
print(f'   Mean[0] sebelum: {X_train.iloc[:,0].mean():.2f} → setelah: {X_train_scaled[:,0].mean():.4f}')

In [ ]:
# Simpan data preprocessed
import os, joblib
os.makedirs('../models', exist_ok=True)

np.save('../models/X_train.npy', X_train_scaled)
np.save('../models/X_val.npy', X_val_scaled)
np.save('../models/X_test.npy', X_test_scaled)
np.save('../models/y_train.npy', y_train.values)
np.save('../models/y_val.npy', y_val.values)
np.save('../models/y_test.npy', y_test.values)

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_target, '../models/label_encoder.pkl')
joblib.dump(le_dict, '../models/feature_encoders.pkl')

# Simpan juga versi unscaled untuk Decision Tree
np.save('../models/X_train_raw.npy', X_train.values)
np.save('../models/X_val_raw.npy', X_val.values)
np.save('../models/X_test_raw.npy', X_test.values)

# Simpan nama fitur
joblib.dump(feature_cols, '../models/feature_cols.pkl')

print('✅ Semua data dan objek preprocessing berhasil disimpan ke folder models/')

## 7. Ringkasan Preprocessing

| Langkah | Teknik | Justifikasi |
|---------|--------|-------------|
| Missing Values | Tidak ada | Dataset sudah bersih |
| Duplikat | Tidak ada | Dataset unik |
| Feature Engineering | Tambah kolom BMI | BMI adalah indikator utama obesitas |
| Encoding | LabelEncoder | Fitur biner & ordinal, hemat dimensi |
| Scaling | StandardScaler | Diperlukan untuk KNN dan algoritma berbasis jarak |
| Split | 60-20-20 | Stratified split untuk menjaga distribusi kelas |